In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Localizar la carpeta data.
possible_paths = [Path("data"), Path("../data")]

DATA_DIR = next(
    path for path in possible_paths
    if (path / "train.csv").exists()
)

# Cargar los archivos.
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

# Dimensiones de los conjuntos.
print("Train:", train.shape)
print("Test:", test.shape)
print("Sample submission:", sample_submission.shape)

# Comprobar el objetivo.
print("\nValores ausentes en SalePrice:")
print(train["SalePrice"].isna().sum())

# Comprobar identificadores.
print("\nIDs duplicados en train:")
print(train["Id"].duplicated().sum())

print("\nIDs duplicados en test:")
print(test["Id"].duplicated().sum())

# Comprobar que train y test contienen las mismas
# características, excluyendo el objetivo.
train_features = set(train.columns) - {"SalePrice"}
test_features = set(test.columns)

print("\nColumnas que faltan en test:")
print(sorted(train_features - test_features))

print("\nColumnas adicionales en test:")
print(sorted(test_features - train_features))

# Primer vistazo a los valores ausentes.
missing = train.isna().sum()
missing = missing[missing > 0].sort_values(
    ascending=False
)

print("\nColumnas con valores ausentes:")
display(missing.to_frame(name="missing_count"))

Train: (1460, 81)
Test: (1459, 80)
Sample submission: (1459, 2)

Valores ausentes en SalePrice:
0

IDs duplicados en train:
0

IDs duplicados en test:
0

Columnas que faltan en test:
[]

Columnas adicionales en test:
[]

Columnas con valores ausentes:


,missing_count
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
FireplaceQu,690
LotFrontage,259
GarageType,81
GarageYrBlt,81
GarageFinish,81


In [2]:
def missing_report(df, name):
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    })

    report = report[report["missing_count"] > 0]
    report = report.sort_values(
        "missing_pct",
        ascending=False
    )

    print(f"\nValores ausentes en {name}:")
    display(report)


missing_report(train, "train")
missing_report(test, "test")


Valores ausentes en train:


,missing_count,missing_pct
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
FireplaceQu,690,47.26
LotFrontage,259,17.74
GarageType,81,5.55
GarageYrBlt,81,5.55
GarageFinish,81,5.55



Valores ausentes en test:


,missing_count,missing_pct
PoolQC,1456,99.79
MiscFeature,1408,96.50
Alley,1352,92.67
Fence,1169,80.12
MasVnrType,894,61.27
FireplaceQu,730,50.03
LotFrontage,227,15.56
GarageQual,78,5.35
GarageCond,78,5.35
GarageYrBlt,78,5.35


In [3]:
# 1. Investigar los valores ausentes relacionados con el garaje.

garage_cols = [
    "GarageType",
    "GarageYrBlt",
    "GarageFinish",
    "GarageCars",
    "GarageArea",
    "GarageQual",
    "GarageCond",
]

display(
    train.loc[
        train["GarageType"].isna(),
        garage_cols
    ].head(10)
)

print(
    "Casas sin GarageType pero con GarageCars > 0:",
    (
        train.loc[train["GarageType"].isna(), "GarageCars"] > 0
    ).sum()
)

print(
    "Casas sin GarageType pero con GarageArea > 0:",
    (
        train.loc[train["GarageType"].isna(), "GarageArea"] > 0
    ).sum()
)


# 2. Investigar el revestimiento de mampostería.

mas_vnr_missing = train[
    ["MasVnrType", "MasVnrArea"]
].isna().any(axis=1)

display(
    train.loc[
        mas_vnr_missing,
        ["Id", "MasVnrType", "MasVnrArea", "SalePrice"]
    ]
)


# 3. Examinar los valores ausentes de LotFrontage.

display(
    train.loc[
        train["LotFrontage"].isna(),
        ["Id", "Neighborhood", "LotArea", "LotFrontage"]
    ].head(10)
)


# 4. Examinar el registro sin información eléctrica.

display(
    train.loc[
        train["Electrical"].isna(),
        ["Id", "Electrical", "CentralAir", "Heating", "SalePrice"]
    ]
)

,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond
39,NaN,NaN,NaN,0,0,NaN,NaN
48,NaN,NaN,NaN,0,0,NaN,NaN
78,NaN,NaN,NaN,0,0,NaN,NaN
88,NaN,NaN,NaN,0,0,NaN,NaN
89,NaN,NaN,NaN,0,0,NaN,NaN
99,NaN,NaN,NaN,0,0,NaN,NaN
108,NaN,NaN,NaN,0,0,NaN,NaN
125,NaN,NaN,NaN,0,0,NaN,NaN
127,NaN,NaN,NaN,0,0,NaN,NaN
140,NaN,NaN,NaN,0,0,NaN,NaN


Casas sin GarageType pero con GarageCars > 0: 0
Casas sin GarageType pero con GarageArea > 0: 0


,Id,MasVnrType,MasVnrArea,SalePrice
1,2,NaN,0.0,181500
3,4,NaN,0.0,140000
5,6,NaN,0.0,143000
8,9,NaN,0.0,129900
9,10,NaN,0.0,118000
...,...,...,...,...
1454,1455,NaN,0.0,185000
1455,1456,NaN,0.0,175000
1457,1458,NaN,0.0,266500
1458,1459,NaN,0.0,142125


,Id,Neighborhood,LotArea,LotFrontage
7,8,NWAmes,10382,NaN
12,13,Sawyer,12968,NaN
14,15,NAmes,10920,NaN
16,17,NAmes,11241,NaN
24,25,Sawyer,8246,NaN
31,32,Sawyer,8544,NaN
42,43,SawyerW,9180,NaN
43,44,CollgCr,9200,NaN
50,51,Gilbert,13869,NaN
64,65,CollgCr,9375,NaN


,Id,Electrical,CentralAir,Heating,SalePrice
1379,1380,NaN,Y,GasA,167500


In [4]:
# Crear máscaras para identificar los valores ausentes.
type_missing = train["MasVnrType"].isna()
area_missing = train["MasVnrArea"].isna()

# Comprobar cómo se relacionan ambas variables.
print(
    "MasVnrType ausente y MasVnrArea = 0:",
    (type_missing & train["MasVnrArea"].eq(0)).sum()
)

print(
    "MasVnrType ausente y MasVnrArea > 0:",
    (type_missing & train["MasVnrArea"].gt(0)).sum()
)

print(
    "Ambas variables ausentes:",
    (type_missing & area_missing).sum()
)

print(
    "MasVnrType presente y MasVnrArea ausente:",
    ((~type_missing) & area_missing).sum()
)

# Mostrar las filas donde falta el área.
display(
    train.loc[
        area_missing,
        ["Id", "MasVnrType", "MasVnrArea", "SalePrice"]
    ]
)

MasVnrType ausente y MasVnrArea = 0: 859
MasVnrType ausente y MasVnrArea > 0: 5
Ambas variables ausentes: 8
MasVnrType presente y MasVnrArea ausente: 0


,Id,MasVnrType,MasVnrArea,SalePrice
234,235,NaN,NaN,216500
529,530,NaN,NaN,200624
650,651,NaN,NaN,205950
936,937,NaN,NaN,184900
973,974,NaN,NaN,182000
977,978,NaN,NaN,199900
1243,1244,NaN,NaN,465000
1278,1279,NaN,NaN,237000


In [5]:
# Identificar viviendas con tipo ausente y área positiva.
mask = (
    train["MasVnrType"].isna()
    & train["MasVnrArea"].gt(0)
)

columns = [
    "Id",
    "MasVnrType",
    "MasVnrArea",
    "Neighborhood",
    "OverallQual",
    "YearBuilt",
    "SalePrice",
]

display(train.loc[mask, columns])

,Id,MasVnrType,MasVnrArea,Neighborhood,OverallQual,YearBuilt,SalePrice
624,625,NaN,288.0,NWAmes,6,1972,165150
773,774,NaN,1.0,NAmes,5,1958,114500
1230,1231,NaN,1.0,Sawyer,5,1977,190000
1300,1301,NaN,344.0,Gilbert,7,1999,225000
1334,1335,NaN,312.0,BrDale,5,1970,125000


In [6]:
# Viviendas en las que faltan tanto el tipo como el área.
both_missing = (
    train["MasVnrType"].isna()
    & train["MasVnrArea"].isna()
)

columns = [
    "Id",
    "Neighborhood",
    "OverallQual",
    "YearBuilt",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "MasVnrArea",
    "SalePrice",
]

display(train.loc[both_missing, columns])

,Id,Neighborhood,OverallQual,YearBuilt,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,SalePrice
234,235,Gilbert,6,2002,VinylSd,VinylSd,NaN,NaN,216500
529,530,Crawfor,6,1957,Wd Sdng,Stone,NaN,NaN,200624
650,651,Somerst,7,2007,CemntBd,CmentBd,NaN,NaN,205950
936,937,SawyerW,7,2003,VinylSd,VinylSd,NaN,NaN,184900
973,974,Somerst,7,2007,CemntBd,CmentBd,NaN,NaN,182000
977,978,Somerst,7,2006,VinylSd,VinylSd,NaN,NaN,199900
1243,1244,NridgHt,10,2006,VinylSd,VinylSd,NaN,NaN,465000
1278,1279,CollgCr,8,2002,VinylSd,VinylSd,NaN,NaN,237000


In [7]:
# 1. Resumen de tipos de datos.
print("=== Tipos de datos ===")
print(train.dtypes.value_counts())

# Ver el tipo, número de valores distintos y valores ausentes
# de cada columna.
dtype_report = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "unique_values": train.nunique(),
    "missing_values": train.isna().sum(),
})

display(dtype_report)


# 2. Comprobar filas duplicadas completas.
print("\n=== Filas duplicadas ===")
print("Train:", train.duplicated().sum())
print("Test:", test.duplicated().sum())


# 3. Comprobar identificadores compartidos.
shared_ids = set(train["Id"]) & set(test["Id"])

print("\n=== Identificadores ===")
print("IDs compartidos entre train y test:", len(shared_ids))


# 4. Comprobar la variable objetivo.
print("\n=== SalePrice ===")
print("Valores no positivos:", (train["SalePrice"] <= 0).sum())
print("Valores no finitos:", (~np.isfinite(train["SalePrice"])).sum())


# 5. Comparar valores ausentes en train y test.
missing_comparison = pd.DataFrame({
    "train_missing": train.isna().sum(),
    "train_pct": (train.isna().mean() * 100).round(2),
    "test_missing": test.isna().sum(),
    "test_pct": (test.isna().mean() * 100).round(2),
}).fillna(0)

missing_comparison = missing_comparison.sort_values(
    ["train_missing", "test_missing"],
    ascending=False,
)

display(missing_comparison.head(25))


# 6. Comprobar que los tipos de las características coinciden.
train_features = train.drop(columns="SalePrice")

dtype_comparison = pd.DataFrame({
    "train_dtype": train_features.dtypes.astype(str),
    "test_dtype": test.dtypes.astype(str),
})

print("\n=== Diferencias de tipos entre train y test ===")
display(
    dtype_comparison[
        dtype_comparison["train_dtype"]
        != dtype_comparison["test_dtype"]
    ]
)

=== Tipos de datos ===
str        43
int64      35
float64     3
Name: count, dtype: int64


,dtype,unique_values,missing_values
Id,int64,1460,0
MSSubClass,int64,15,0
MSZoning,str,5,0
LotFrontage,float64,110,259
LotArea,int64,1073,0
...,...,...,...
MoSold,int64,12,0
YrSold,int64,5,0
SaleType,str,9,0
SaleCondition,str,6,0



=== Filas duplicadas ===
Train: 0
Test: 0

=== Identificadores ===
IDs compartidos entre train y test: 0

=== SalePrice ===
Valores no positivos: 0
Valores no finitos: 0


,train_missing,train_pct,test_missing,test_pct
PoolQC,1453,99.52,1456.0,99.79
MiscFeature,1406,96.30,1408.0,96.50
Alley,1369,93.77,1352.0,92.67
Fence,1179,80.75,1169.0,80.12
MasVnrType,872,59.73,894.0,61.27
FireplaceQu,690,47.26,730.0,50.03
LotFrontage,259,17.74,227.0,15.56
GarageCond,81,5.55,78.0,5.35
GarageFinish,81,5.55,78.0,5.35
GarageQual,81,5.55,78.0,5.35



=== Diferencias de tipos entre train y test ===


,train_dtype,test_dtype
BsmtFinSF1,int64,float64
BsmtFinSF2,int64,float64
BsmtUnfSF,int64,float64
TotalBsmtSF,int64,float64
BsmtFullBath,int64,float64
BsmtHalfBath,int64,float64
GarageCars,int64,float64
GarageArea,int64,float64


In [8]:
# 1. Resumen de las variables numéricas.
numeric_cols = train.select_dtypes(include=np.number).columns

numeric_summary = train[numeric_cols].describe().T

display(
    numeric_summary[
        ["min", "25%", "50%", "75%", "max"]
    ]
)


# 2. Buscar valores negativos.
negative_train = (
    train.select_dtypes(include=np.number) < 0
).sum()

negative_test = (
    test.select_dtypes(include=np.number) < 0
).sum()

print("Valores negativos en train:")
display(negative_train[negative_train > 0])

print("Valores negativos en test:")
display(negative_test[negative_test > 0])


# 3. Comprobar coherencia de los años.
year_checks = {
    "YearBuilt > YrSold":
        train["YearBuilt"] > train["YrSold"],

    "YearRemodAdd > YrSold":
        train["YearRemodAdd"] > train["YrSold"],

    "YearRemodAdd < YearBuilt":
        train["YearRemodAdd"] < train["YearBuilt"],

    "GarageYrBlt > YrSold":
        train["GarageYrBlt"] > train["YrSold"],
}

print("Comprobaciones de coherencia temporal:")

for name, condition in year_checks.items():
    print(f"{name}: {condition.sum()} casos")


# 4. Comprobar los meses de venta.
invalid_months = ~train["MoSold"].between(1, 12)

print(
    "Valores de MoSold fuera del rango 1-12:",
    invalid_months.sum()
)

,min,25%,50%,75%,max
Id,1.0,365.75,730.5,1095.25,1460.0
MSSubClass,20.0,20.00,50.0,70.00,190.0
LotFrontage,21.0,59.00,69.0,80.00,313.0
LotArea,1300.0,7553.50,9478.5,11601.50,215245.0
OverallQual,1.0,5.00,6.0,7.00,10.0
OverallCond,1.0,5.00,5.0,6.00,9.0
YearBuilt,1872.0,1954.00,1973.0,2000.00,2010.0
YearRemodAdd,1950.0,1967.00,1994.0,2004.00,2010.0
MasVnrArea,0.0,0.00,0.0,166.00,1600.0
BsmtFinSF1,0.0,0.00,383.5,712.25,5644.0


Valores negativos en train:


Series([], dtype: int64)

Valores negativos en test:


Series([], dtype: int64)

Comprobaciones de coherencia temporal:
YearBuilt > YrSold: 0 casos
YearRemodAdd > YrSold: 1 casos
YearRemodAdd < YearBuilt: 0 casos
GarageYrBlt > YrSold: 0 casos
Valores de MoSold fuera del rango 1-12: 0
